# 03 — Model training and evaluation

Compare candidate classifiers with a majority-class baseline. Model selection uses training-only repeated stratified cross-validation; the test set is reserved for final evaluation. If no candidate clears the pre-declared screen, recommend no model.


In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed').exists() and (ROOT.parent / 'data' / 'processed').exists(): ROOT = ROOT.parent
path = ROOT / 'data' / 'processed' / 'modeling_dataset.csv'
if not path.exists(): raise FileNotFoundError('Run notebook 02 first; expected data/processed/modeling_dataset.csv')
data = pd.read_csv(path)
excluded = {'target_won','deal_stage','close_value','close_date','engage_date','opportunity_id','account','product_join_key'}
features = [c for c in data.columns if c not in excluded and not c.endswith('_id') and data[c].nunique(dropna=True) > 1]
X, y = data[features].copy(), data['target_won'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
numeric = X_train.select_dtypes(include=np.number).columns.tolist()
categorical = [c for c in X_train.columns if c not in numeric]
preprocessor = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical)])
models = {'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced'), 'Random Forest': RandomForestClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1), 'Extra Trees': ExtraTreesClassifier(n_estimators=200, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1)}
try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(n_estimators=120, max_depth=3, learning_rate=0.05, subsample=0.85, colsample_bytree=0.85, eval_metric='logloss', random_state=42, n_jobs=2)
except ImportError: print('XGBoost unavailable; skipped.')
try:
    from catboost import CatBoostClassifier
    models['CatBoost'] = CatBoostClassifier(iterations=120, depth=5, learning_rate=0.05, verbose=False, random_seed=42, thread_count=2, allow_writing_files=False)
except ImportError: print('CatBoost unavailable; skipped.')
try:
    from lightgbm import LGBMClassifier
    models['LightGBM'] = LGBMClassifier(n_estimators=120, learning_rate=0.05, verbosity=-1, random_state=42, n_jobs=2)
except ImportError: print('LightGBM unavailable; skipped.')
print(f'Train={len(X_train):,}; test={len(X_test):,}; features={len(features)}')


## Training-only cross-validation

A candidate is eligible for held-out evaluation only when mean CV ROC-AUC exceeds 0.52 and its standard deviation is below 0.10. This is a conservative project screening rule, not a universal statistical threshold.


In [ ]:
cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=2, random_state=42)
cv_rows, pipelines = [], {}
for name, estimator in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', estimator)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=1)
    cv_rows.append({'model':name, 'cv_roc_auc_mean':scores.mean(), 'cv_roc_auc_std':scores.std()})
    pipelines[name] = pipe
cv_results = pd.DataFrame(cv_rows).sort_values('cv_roc_auc_mean', ascending=False).reset_index(drop=True)
display(cv_results.style.format({'cv_roc_auc_mean':'{:.3f}', 'cv_roc_auc_std':'{:.3f}'}))
eligible = cv_results[(cv_results.cv_roc_auc_mean > 0.52) & (cv_results.cv_roc_auc_std < 0.10)]
if eligible.empty: print('DECISION: No model recommended. No candidate passed the pre-declared training-only signal screen.')
else: print('Candidates eligible for final test evaluation:', eligible.model.tolist())


## Held-out evaluation

The majority-class baseline is always reported. The test partition is not used to choose among models.


In [ ]:
baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
rows = []
def score_model(name, estimator):
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    prob = estimator.predict_proba(X_test)[:, 1] if hasattr(estimator, 'predict_proba') else None
    auc = roc_auc_score(y_test, prob) if prob is not None else np.nan
    return {'model':name, 'accuracy':accuracy_score(y_test,pred), 'balanced_accuracy':balanced_accuracy_score(y_test,pred), 'precision_won':precision_score(y_test,pred,zero_division=0), 'recall_won':recall_score(y_test,pred,zero_division=0), 'f1_won':f1_score(y_test,pred,zero_division=0), 'roc_auc':auc}, pred
base_row, _ = score_model('Majority-class baseline', baseline); rows.append(base_row)
if not eligible.empty:
    for name in eligible.model:
        row, pred = score_model(name, pipelines[name]); rows.append(row)
        print(f'\\n{name}\\n', classification_report(y_test, pred, target_names=['Lost','Won'], zero_division=0))
        sns.heatmap(confusion_matrix(y_test,pred), annot=True, fmt='d', cmap='Blues', xticklabels=['Lost','Won'], yticklabels=['Lost','Won'])
        plt.title(f'{name} — held-out confusion matrix'); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.tight_layout(); plt.show()
else:
    print('No candidate passed cross-validation screening; only the baseline is evaluated. No model is promoted.')
results = pd.DataFrame(rows)
display(results.style.format(precision=3))

report_dir = ROOT / 'reports'; report_dir.mkdir(parents=True, exist_ok=True)
summary = 'No model recommended: no candidate passed the training-only screen.' if eligible.empty else 'Candidate(s) passed the training-only screen; review held-out results before drawing conclusions.'
report = '# Win/loss modeling evaluation\\n\\n' + summary + '\\n\\n## Training-only cross-validation\\n\\n' + cv_results.to_string(index=False) + '\\n\\n## Held-out test metrics\\n\\n' + results.to_string(index=False) + '\\n\\n## Limitations\\n\\nRetrospective analysis of synthetic CRM data; not suitable for live deal scoring.\\n'
(report_dir / 'ml_evaluation_report.md').write_text(report, encoding='utf-8')
print('Report saved. No model artifact is saved automatically.')
